In [6]:
import torch
import json
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm


print("Libraries imported successfully.")

Libraries imported successfully.


MODEL LOAD

In [1]:
from transformers import CLIPProcessor, CLIPModel

model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name)

print("CLIP model loaded successfully!")

d:\Pinterest-AI-Recommender\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Pinterest-AI-Recommender\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\musam\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate d

CLIP model loaded successfully!


TESTING

In [3]:

# Project ke Data/images folder ka path
images_folder = Path(r"D:\Pinterest-AI-Recommender\Data\images")

# Folder se pehli image find karo
image_files = list(images_folder.glob("*.jpg"))

print("Images found:", len(image_files))

# Pehli image load karo
image_path = image_files[0]
image = Image.open(image_path).convert("RGB")

print("Image:", image_path.name)
print("Image size:", image.size)

Images found: 14161
Image: aesthetic_stock_00045.jpg
Image size: (1200, 800)


In [5]:
# Image ko CLIP ke input format mein prepare karna
inputs = processor(images=image, return_tensors="pt")

# CLIP se image features nikalna
with torch.no_grad():
    outputs = model.get_image_features(**inputs)

# Output object se actual tensor lena
image_features = outputs.pooler_output

print("Embedding shape:", image_features.shape)

Embedding shape: torch.Size([1, 512])


IMAGE EMBEDDING

In [8]:

# Project paths
PROJECT_ROOT = Path(r"D:\Pinterest-AI-Recommender")
IMAGES_DIR = PROJECT_ROOT / "Data" / "images"
METADATA_PATH = PROJECT_ROOT / "Data" / "metadata.csv"
MODELS_DIR = PROJECT_ROOT / "Models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Sirf dataset mein maujood image files select karo
extensions = {".jpg", ".jpeg", ".png", ".webp"}
image_paths = sorted(
    p for p in IMAGES_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in extensions
)

all_embeddings = []
valid_filenames = []
batch_size = 32

model.eval()

for start in tqdm(range(0, len(image_paths), batch_size)):
    batch_paths = image_paths[start:start + batch_size]
    batch_images = []
    batch_names = []

    for path in batch_paths:
        try:
            with Image.open(path) as img:
                batch_images.append(img.convert("RGB"))
                batch_names.append(path.name)
        except Exception:
            continue

    if not batch_images:
        continue

    inputs = processor(images=batch_images, return_tensors="pt")

    with torch.no_grad():
        outputs = model.get_image_features(**inputs)

        # Tumhare Transformers version ke output ko handle karo
        features = (
            outputs.pooler_output
            if hasattr(outputs, "pooler_output")
            else outputs
        )

        features = torch.nn.functional.normalize(features, p=2, dim=1)

    all_embeddings.append(features.cpu())
    valid_filenames.extend(batch_names)

if not all_embeddings:
    raise RuntimeError("Koi valid image process nahi hui.")

embeddings = torch.cat(all_embeddings, dim=0)

torch.save(
    {
        "embeddings": embeddings,
        "filenames": valid_filenames,
    },
    MODELS_DIR / "clip_image_embeddings.pt",
)

with open(MODELS_DIR / "clip_image_filenames.json", "w", encoding="utf-8") as f:
    json.dump(valid_filenames, f, ensure_ascii=False, indent=2)

print("Embedding generation complete!")
print("Images processed:", len(valid_filenames))
print("Embedding shape:", tuple(embeddings.shape))
print("Saved to:", MODELS_DIR / "clip_image_embeddings.pt")

100%|██████████| 443/443 [45:26<00:00,  6.16s/it] 

Embedding generation complete!
Images processed: 14161
Embedding shape: (14161, 512)
Saved to: D:\Pinterest-AI-Recommender\Models\clip_image_embeddings.pt
